[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](#)

# Likelihood_Error_Analyzer_Multi_Models

This notebook computes a **Likelihood of Error Score (0–5)** for each record in **Job_Classifications_Batch.json** by comparing only these inputs:

- `Job_Classifications_Batch.json`
- `alternative_roles_analysis.json`
- `Role_Confusion_Crosswalk.json`
- `Universal_Role_Classification_Prompt.json`

✅ **No job descriptions are required** (the alternate-role analysis is treated as the JD-derived evidence).


In [ ]:
# ==== 0) Install dependencies (Colab) ====
# If you are running locally, you can comment this out.
!pip -q install pandas numpy matplotlib transformers accelerate sentencepiece


In [ ]:
# ==== 9) Multi-model setup (API + local open-weights) — removed Google Google model support ====
# This section is OPTIONAL. The base Likelihood score is computed deterministically from your JSON files.
# Use this only if you want an LLM "judge" to add/validate *record-level* signals (e.g., hedging language, title-only reasoning).

# -------------------------
# A) Choose your mode
# -------------------------
# Modes:
#   - "LOCAL_TRANSFORMERS"  : run an open-weight model locally in Colab (GPU recommended)
#   - "HF_INFERENCE_API"    : Hugging Face Inference API (serverless) via token
#   - "MISTRAL_API"         : Mistral API (optional)
#   - "ANTHROPIC_API"       : Anthropic API (optional)
#
# NOTE: Google model code intentionally removed per your request.

MODEL_MODE = "LOCAL_TRANSFORMERS"   # <-- change me

# -------------------------
# B) Model selection (local / HF)
# -------------------------
SUPPORTED_LOCAL_MODELS = {
    # Requested additions:
    "phi3_medium_128k": "microsoft/Phi-3-medium-128k-instruct",
    "deepseek_r1_distill_qwen_8b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-8B",

    # Other common open-weight options you may want:
    "qwen2_5_7b": "Qwen/Qwen2.5-7B-Instruct",
    "qwen2_5_14b": "Qwen/Qwen2.5-14B-Instruct",
    "llama3_1_8b": "meta-llama/Llama-3.1-8B-Instruct",
    "llama3_1_70b": "meta-llama/Llama-3.1-70B-Instruct",
}

MODEL_KEY = "phi3_medium_128k"  # <-- change me
MODEL_ID = SUPPORTED_LOCAL_MODELS[MODEL_KEY]

# -------------------------
# C) API keys (only needed for API modes)
# -------------------------
# Prefer Colab Secrets. You can also paste them here.
HF_TOKEN = ""           # Hugging Face token (for HF_INFERENCE_API)
MISTRAL_API_KEY = ""    # for MISTRAL_API
ANTHROPIC_API_KEY = ""  # for ANTHROPIC_API

# -------------------------
# D) Local model loader
# -------------------------
import os
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = None
model = None

def load_local_model(model_id: str, use_4bit: bool = True):
    """
    Load an open-weight model in Colab.
    - use_4bit=True reduces VRAM; requires bitsandbytes.
    """
    global tokenizer, model

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)

    kwargs = dict(
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

    if use_4bit and torch.cuda.is_available():
        try:
            from transformers import BitsAndBytesConfig
            bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            kwargs["quantization_config"] = bnb
        except Exception as e:
            print(f"⚠️ 4-bit not available (bitsandbytes). Loading without quantization. ({e})")

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    print(f"✅ Local model loaded: {model_id}")

# -------------------------
# E) Unified generate() function
# -------------------------
def _format_chat_prompt(system: str, user: str) -> str:
    # Works for most instruct models. For chat-template models, we'll try apply_chat_template if available.
    return f"{system.strip()}\n\nUSER:\n{user.strip()}\n\nASSISTANT:\n"

def generate_text(system_prompt: str, user_prompt: str, max_new_tokens: int = 400, temperature: float = 0.2):
    """
    Returns model text completion.
    """
    if MODEL_MODE == "LOCAL_TRANSFORMERS":
        if tokenizer is None or model is None:
            raise RuntimeError("Local model not loaded. Run load_local_model(MODEL_ID) first.")

        # Prefer native chat template if present
        try:
            messages = [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}]
            input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
        except Exception:
            prompt = _format_chat_prompt(system_prompt, user_prompt)
            input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

        with torch.no_grad():
            out = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=(temperature > 0),
                temperature=temperature,
                eos_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0], skip_special_tokens=True)

        # Best-effort: return only the assistant portion
        if "ASSISTANT:" in text:
            text = text.split("ASSISTANT:", 1)[-1].strip()
        return text.strip()

    elif MODEL_MODE == "HF_INFERENCE_API":
        if not HF_TOKEN:
            raise RuntimeError("HF_TOKEN is empty. Set it (Colab Secrets recommended).")
        try:
            from huggingface_hub import InferenceClient
        except ImportError:
            raise ImportError("Please install huggingface_hub: !pip -q install huggingface_hub")

        client = InferenceClient(model=MODEL_ID, token=HF_TOKEN)
        prompt = _format_chat_prompt(system_prompt, user_prompt)
        return client.text_generation(
            prompt=prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            return_full_text=False,
        ).strip()

    elif MODEL_MODE == "MISTRAL_API":
        if not MISTRAL_API_KEY:
            raise RuntimeError("MISTRAL_API_KEY is empty.")
        import requests
        url = "https://api.mistral.ai/v1/chat/completions"
        headers = {"Authorization": f"Bearer {MISTRAL_API_KEY}", "Content-Type":"application/json"}
        payload = {
            "model": "mistral-large-latest",
            "messages": [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}],
            "temperature": temperature,
            "max_tokens": max_new_tokens,
        }
        r = requests.post(url, headers=headers, json=payload, timeout=120)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip()

    elif MODEL_MODE == "ANTHROPIC_API":
        if not ANTHROPIC_API_KEY:
            raise RuntimeError("ANTHROPIC_API_KEY is empty.")
        import requests
        url = "https://api.anthropic.com/v1/messages"
        headers = {
            "x-api-key": ANTHROPIC_API_KEY,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        }
        payload = {
            "model": "claude-3-5-sonnet-latest",
            "max_tokens": max_new_tokens,
            "temperature": temperature,
            "system": system_prompt,
            "messages": [{"role":"user","content": user_prompt}],
        }
        r = requests.post(url, headers=headers, json=payload, timeout=120)
        r.raise_for_status()
        return r.json()["content"][0]["text"].strip()

    else:
        raise ValueError(f"Invalid MODEL_MODE: {MODEL_MODE}")

print("✅ Multi-model adapters ready.")
print("MODE:", MODEL_MODE)
print("MODEL_KEY:", MODEL_KEY)
print("MODEL_ID:", MODEL_ID)


In [ ]:
# ==== 1) Upload inputs (ZIP or individual JSON files) ====
# Option A: Upload a zip named exactly: "Likelihood Evaluation Resources.zip" containing the 4 JSON files below.
# Option B: Upload the 4 JSON files directly.

from google.colab import files
import os, zipfile, glob

uploaded = files.upload()

ZIP_NAME = "Likelihood Evaluation Resources.zip"
WORKDIR = "/content/likelihood_eval"
os.makedirs(WORKDIR, exist_ok=True)

# If ZIP uploaded, extract it into WORKDIR
if ZIP_NAME in uploaded:
    zip_path = os.path.join("/content", ZIP_NAME)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(WORKDIR)
    print(f"✅ Extracted {ZIP_NAME} to {WORKDIR}")

# Move any individually uploaded files into WORKDIR (including the zip itself, harmless)
for fn in uploaded.keys():
    src = os.path.join("/content", fn)
    dst = os.path.join(WORKDIR, fn)
    if os.path.exists(src) and src != dst:
        os.replace(src, dst)

print(f"✅ Working directory: {WORKDIR}")
print("Files found:", [os.path.basename(p) for p in glob.glob(os.path.join(WORKDIR, '*'))])

def find_file(candidates):
    cand_lower = [c.lower() for c in candidates]
    # direct match
    for c in candidates:
        p = os.path.join(WORKDIR, c)
        if os.path.exists(p):
            return p
    # case-insensitive basename match
    for p in glob.glob(os.path.join(WORKDIR, "*")):
        if os.path.basename(p).lower() in cand_lower:
            return p
    raise FileNotFoundError(f"Could not find any of: {candidates} in {WORKDIR}")

PATH_ALT       = find_file(["alternative_roles_analysis.json"])
PATH_JOB_BATCH = find_file(["Job_Classifications_Batch.json"])
PATH_CROSSWALK = find_file(["Role_Confusion_Crosswalk.json"])
PATH_PROMPT    = find_file(["Universal_Role_Classification_Prompt.json"])

print("✅ Using:")
print(" - alternative_roles_analysis:", PATH_ALT)
print(" - Job_Classifications_Batch :", PATH_JOB_BATCH)
print(" - Role_Confusion_Crosswalk  :", PATH_CROSSWALK)
print(" - Universal prompt          :", PATH_PROMPT)


In [ ]:
# ==== 2) Load JSON files into DataFrames ====
import json
import pandas as pd
import numpy as np

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

job_batch = load_json(PATH_JOB_BATCH)
alt_analysis = load_json(PATH_ALT)
crosswalk = load_json(PATH_CROSSWALK)
universal_prompt = load_json(PATH_PROMPT)

# The provided files are expected to be lists of rows for the first 3
df_jobs = pd.DataFrame(job_batch if isinstance(job_batch, list) else job_batch.get("rows", []))
df_alt  = pd.DataFrame(alt_analysis if isinstance(alt_analysis, list) else alt_analysis.get("rows", []))
df_cross = pd.DataFrame(crosswalk if isinstance(crosswalk, list) else crosswalk.get("rows", []))

print("df_jobs :", df_jobs.shape)
print("df_alt  :", df_alt.shape)
print("df_cross:", df_cross.shape)

display(df_jobs.head(3))


In [ ]:
# ==== 3) Key fields + safe normalization ====
import re

def norm(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    return str(s).strip()

# Job batch: find title + major role group
title_cols = [c for c in df_jobs.columns if c.lower() in ["job_title_original","job title","job_title","title","new_job_title"]]
role_cols  = [c for c in df_jobs.columns if c.lower() in ["major_role_group","major role group","major_role","major"]]

if not title_cols or not role_cols:
    raise KeyError(f"Could not find job title / major role columns. Columns found: {list(df_jobs.columns)}")

TITLE_COL = title_cols[0]
ROLE_COL  = role_cols[0]

df_jobs["job_title_key"] = df_jobs[TITLE_COL].map(norm)
df_jobs["major_role_group"] = df_jobs[ROLE_COL].map(norm)

# Alternative analysis: prefer Job Code linkage if present, else title
ALT_JOB_CODE_COL = None
for c in df_alt.columns:
    if c.lower().replace(" ", "") in ["jobcode","job_code","jobcodenumber"]:
        ALT_JOB_CODE_COL = c
        break

ALT_TITLE_COL = None
for c in df_alt.columns:
    if c.lower() in ["job_title_original","job title","job_title","title","job_title_key"]:
        ALT_TITLE_COL = c
        break

print("Using TITLE_COL =", TITLE_COL)
print("Using ROLE_COL  =", ROLE_COL)
print("ALT_JOB_CODE_COL =", ALT_JOB_CODE_COL)
print("ALT_TITLE_COL    =", ALT_TITLE_COL)


In [ ]:
# ==== 4) Build crosswalk priors (role-level risk) ====

# Try to find the crosswalk's role column
cross_role_cols = [c for c in df_cross.columns if c.lower() in ["major_role_group","major role group","major role","role","group"]]
if not cross_role_cols:
    raise KeyError(f"Could not find a role column in crosswalk. Columns: {list(df_cross.columns)}")
CROSS_ROLE_COL = cross_role_cols[0]

# Locate key numeric fields if present
def find_col(candidates):
    for cand in candidates:
        for c in df_cross.columns:
            if c.lower() == cand.lower():
                return c
    # fuzzy contains
    for cand in candidates:
        for c in df_cross.columns:
            if cand.lower() in c.lower():
                return c
    return None

COL_ERR = find_col(["Human_Error_Probability_%","Human Error Probability %","human_error_probability","error_probability"])
COL_RISK = find_col(["Confusion Risk Score","Confusion_Risk_Score","confusion_risk_score"])
COL_MIS = find_col(["Most_Likely_Misclassification","Most Likely Misclassification","most_likely_misclassification"])

if COL_ERR is None or COL_RISK is None or COL_MIS is None:
    print("⚠️ Crosswalk column mapping:")
    print(" - role:", CROSS_ROLE_COL)
    print(" - error%:", COL_ERR)
    print(" - risk:", COL_RISK)
    print(" - likely misclass:", COL_MIS)
    raise KeyError("Crosswalk is missing one or more required columns (error%, risk score, likely misclassification).")

df_cross["major_role_group"] = df_cross[CROSS_ROLE_COL].map(norm)
df_cross["human_error_probability"] = pd.to_numeric(df_cross[COL_ERR], errors="coerce")
df_cross["confusion_risk_score"] = pd.to_numeric(df_cross[COL_RISK], errors="coerce")
df_cross["most_likely_misclassification"] = df_cross[COL_MIS].map(norm)

priors = df_cross[["major_role_group","human_error_probability","confusion_risk_score","most_likely_misclassification"]].dropna(subset=["major_role_group"])
priors = priors.drop_duplicates("major_role_group", keep="first")

print("Priors rows:", priors.shape[0])
display(priors.head(10))


In [ ]:
# ==== 5) Build record-level ambiguity signals from alternative_roles_analysis ====
# We do NOT re-read job descriptions. We only use whatever the alternate-role analyzer already produced.

def to_list(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [norm(v) for v in x if norm(v)]
    s = str(x).strip()
    if not s:
        return []
    # split by commas/semicolons/slashes/newlines
    parts = re.split(r"[;,/\n]+", s)
    return [norm(p) for p in parts if norm(p)]

# Detect alt roles list column
alt_list_col = None
for c in df_alt.columns:
    cl = c.lower()
    if "other plausible" in cl or "other_plausible" in cl or "alternative" in cl or "alternatives" in cl or "other roles" in cl or "other_roles" in cl:
        alt_list_col = c
        break

if alt_list_col is None:
    raise KeyError(f"Could not detect an alternate-roles list column in alternative_roles_analysis. Columns: {list(df_alt.columns)}")

df_alt["alt_roles_list"] = df_alt[alt_list_col].apply(to_list)
df_alt["alt_count"] = df_alt["alt_roles_list"].apply(len)

# Link key: Job Code preferred if present in both; else title
JOB_CODE_COL_JOBS = None
for c in df_jobs.columns:
    if c.lower().replace(" ", "") in ["jobcode","job_code","jobcodenumber"]:
        JOB_CODE_COL_JOBS = c
        break

if JOB_CODE_COL_JOBS and ALT_JOB_CODE_COL:
    df_jobs["job_code_key"] = df_jobs[JOB_CODE_COL_JOBS]
    df_alt["job_code_key"] = df_alt[ALT_JOB_CODE_COL]
    alt_small = df_alt[["job_code_key","alt_roles_list","alt_count"]].copy()
    JOIN_KEY = "job_code_key"
else:
    alt_small = df_alt[[ALT_TITLE_COL, "alt_roles_list", "alt_count"]].copy() if ALT_TITLE_COL else df_alt[["alt_roles_list","alt_count"]].copy()
    if ALT_TITLE_COL:
        alt_small = alt_small.rename(columns={ALT_TITLE_COL: "job_title_key"})
    else:
        # fallback: create empty title key so merge won't explode
        alt_small["job_title_key"] = ""
    JOIN_KEY = "job_title_key"

print("✅ Alt join key:", JOIN_KEY)
display(alt_small.head(5))


In [ ]:
# ==== 6) Critical confusion patterns (from universal prompt) ====
# Since we are not using full job description text, pattern detection is based on *role pairs* that are known to be high-risk.

CRITICAL_PAIRS = {
    ("Analyst","Auditor"),
    ("Auditor","Analyst"),
    ("Manager","Director"),
    ("Director","Manager"),
    ("Technician","Operator"),
    ("Operator","Technician"),
    ("Technician","Mechanic"),
    ("Mechanic","Technician"),
    ("Technician","Skilled Laborer"),
    ("Skilled Laborer","Technician"),
    ("Coordinator","Manager"),
    ("Manager","Coordinator"),
}

def pattern_hit(pred_role, likely_mis):
    a, b = norm(pred_role), norm(likely_mis)
    if not a or not b:
        return 0
    return 1 if (a,b) in CRITICAL_PAIRS else 0

print("Critical pairs loaded:", len(CRITICAL_PAIRS))


In [ ]:
# ==== 7) Compute Likelihood of Error Score (0–5) ====

# Prepare priors from crosswalk
# Expected columns in crosswalk:
# - Role (or similar) for predicted role key
# - Human_Error_Probability_% (or similar)
# - Confusion Risk Score (0..2)
# - Most_Likely_Misclassification
# - Top Match Role (optional, used for confirmation)

# Normalize crosswalk role column
role_key_col = None
for c in df_cross.columns:
    if c.lower() in ["role","major_role_group","major role group","classified role"]:
        role_key_col = c
        break
if role_key_col is None:
    raise KeyError(f"Could not find role key column in crosswalk. Columns: {list(df_cross.columns)}")

df_cross["_role_key"] = df_cross[role_key_col].map(norm)

def pick_col(possible_names):
    for name in possible_names:
        for c in df_cross.columns:
            if c.lower() == name.lower():
                return c
    return None

col_err = pick_col(["Human_Error_Probability_%","Human Error Probability %","human_error_probability"])
col_score = pick_col(["Confusion Risk Score","confusion_risk_score"])
col_most = pick_col(["Most_Likely_Misclassification","most_likely_misclassification"])
col_topmatch = pick_col(["Top Match Role","top_match_role"])

priors = pd.DataFrame({
    "major_role_group": df_cross["_role_key"],
    "human_error_probability": pd.to_numeric(df_cross[col_err], errors="coerce") if col_err else np.nan,
    "confusion_risk_score": pd.to_numeric(df_cross[col_score], errors="coerce") if col_score else np.nan,
    "most_likely_misclassification": df_cross[col_most].map(norm) if col_most else "",
    "top_match_role": df_cross[col_topmatch].map(norm) if col_topmatch else ""
}).drop_duplicates(subset=["major_role_group"])

# Merge job results + priors + alt signals
if JOIN_KEY == "job_code_key":
    df = df_jobs.merge(priors, on="major_role_group", how="left").merge(alt_small, on="job_code_key", how="left")
else:
    df = df_jobs.merge(priors, on="major_role_group", how="left").merge(alt_small, on="job_title_key", how="left")

# Defaults if crosswalk missing for a role
df["human_error_probability"] = df["human_error_probability"].fillna(25.0)  # conservative default
df["confusion_risk_score"] = df["confusion_risk_score"].fillna(1.0)
df["most_likely_misclassification"] = df["most_likely_misclassification"].fillna("")
df["top_match_role"] = df["top_match_role"].fillna("")

df["alt_roles_list"] = df["alt_roles_list"].apply(lambda x: x if isinstance(x, list) else [])
df["alt_count"] = df["alt_count"].fillna(0).astype(int)

# Pattern hit uses predicted role + most-likely misclassification from crosswalk
df["pattern_hit"] = df.apply(lambda r: pattern_hit(r["major_role_group"], r["most_likely_misclassification"]), axis=1)

# Crosswalk confirmation: does crosswalk's top-match role show up among this record's plausible alternatives?
def confirm_crosswalk(row):
    tm = norm(row.get("top_match_role",""))
    if not tm:
        return 0
    alts = row.get("alt_roles_list", [])
    alts_norm = [norm(a) for a in alts]
    return 1 if tm in alts_norm else 0

df["crosswalk_confirmed"] = df.apply(confirm_crosswalk, axis=1)

# Normalize components to 0..1
P = (df["human_error_probability"] / 100).clip(0,1)
C = (df["confusion_risk_score"] / 2).clip(0,1)     # assuming 0..2
A = (df["alt_count"] / 4).clip(0,1)                # cap at 4 alternatives
R = df["pattern_hit"].clip(0,1)
M = df["crosswalk_confirmed"].clip(0,1)            # confirmation signal

# Weighted error probability -> score (adds confirmation term)
# We keep the original structure but "confirm" crosswalk risk with record evidence:
# - If the crosswalk's top-match role is also plausible for this record, risk increases.
df["p_error"] = (0.40*P + 0.18*C + 0.22*A + 0.08*R + 0.12*M).clip(0,1)
df["likelihood_error_score_0_5"] = (5 * df["p_error"]).round(1)

# Helpful labeling
df["likelihood_band"] = pd.cut(
    df["likelihood_error_score_0_5"],
    bins=[-0.001, 1, 2, 3, 4, 5.001],
    labels=["Very Low","Low","Moderate","High","Very High"]
)

cols_out = [
    "job_title_key",
    "major_role_group",
    "likelihood_error_score_0_5",
    "likelihood_band",
    "human_error_probability",
    "confusion_risk_score",
    "most_likely_misclassification",
    "top_match_role",
    "crosswalk_confirmed",
    "alt_count",
    "alt_roles_list",
    "pattern_hit"
]
display(df[cols_out].head(10))
print("✅ Scored records:", len(df))


In [ ]:
# ==== 8) Updated visuals (matplotlib) ====
import matplotlib.pyplot as plt

# 8.1 Score distribution
plt.figure()
df["likelihood_error_score_0_5"].plot(kind="hist", bins=20, title="Likelihood of Error Score (0–5) توزيع")
plt.xlabel("Score (0–5)")
plt.ylabel("Count")
plt.show()

# 8.2 Avg score by predicted role (top 15 by volume)
role_counts = df["major_role_group"].value_counts()
top_roles = role_counts.head(15).index
role_means = df[df["major_role_group"].isin(top_roles)].groupby("major_role_group")["likelihood_error_score_0_5"].mean().sort_values(ascending=False)

plt.figure()
role_means.plot(kind="bar", title="Average Likelihood Score by Predicted Role (Top 15 by Volume)")
plt.xlabel("Predicted major_role_group")
plt.ylabel("Avg score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 8.3 Scatter: alt_count vs score
plt.figure()
plt.scatter(df["alt_count"], df["likelihood_error_score_0_5"])
plt.title("Alternate Roles Count vs Likelihood of Error Score")
plt.xlabel("alt_count (from alternative_roles_analysis)")
plt.ylabel("score (0–5)")
plt.tight_layout()
plt.show()

# 8.4 "Likely misclassification" funnel (top 20)
pair = df.copy()
pair["likely_pair"] = pair["major_role_group"].fillna("") + " → " + pair["most_likely_misclassification"].fillna("")
top_pairs = pair["likely_pair"].value_counts().head(20)

plt.figure()
top_pairs.sort_values().plot(kind="barh", title="Top 20 Predicted → Most-Likely Misclassification Pairs (Crosswalk)")
plt.xlabel("Count")
plt.tight_layout()
plt.show()


## Optional: Multi-model “adjudication” (no job descriptions)

If you want to use open-weight models as an *additional judge*, this notebook supports a plug-in interface.
**Important:** the LLM is only given **(a)** the predicted role, **(b)** crosswalk priors, and **(c)** the alternate-role analysis outputs — not the full JD text.


In [ ]:
# ==== 10) LLM judge prompt + runner (subset) ====
# Use this if you want the model to assess *textual confidence signals* from grouping_justification
# and optionally refine/validate your deterministic signals. This does NOT read job descriptions.

import json
import re
import pandas as pd

JUDGE_SYSTEM = """You are an auditing assistant for an HR job classification evaluation pipeline.
You will be given a JSON record that includes:
- job_title_original
- major_role_group (the chosen classification)
- grouping_justification (may be empty)
- crosswalk signals (overall risk, top-match)
- alt-role evidence (list of plausible alternatives)

Your job:
1) Detect whether the justification is weak, hedge-heavy, or title-only.
2) Detect whether the justification text strongly suggests the competing role (if provided).
3) Output ONLY valid JSON matching the schema below.

Schema:
{
  "hedging_language": true/false,
  "title_only_reasoning": true/false,
  "mentions_competing_role_terms": true/false,
  "notes": "short explanation (<=200 chars)"
}
"""

HEDGE_PAT = re.compile(r"\b(aligns most closely|appears|seems|while|although|likely|generally)\b", re.I)

def judge_record(row: dict, competing_terms=None):
    just = (row.get("grouping_justification") or "").strip()
    hedging = bool(HEDGE_PAT.search(just)) if just else False

    # naive "title-only": justification mentions the title or role but lacks function words
    title_only = False
    if just:
        # if it contains "because the title" or similar
        if re.search(r"\b(title says|because the title|job title)\b", just, re.I):
            title_only = True

    mentions_competing = False
    if just and competing_terms:
        for t in competing_terms:
            if t and re.search(r"\b" + re.escape(t) + r"\b", just, re.I):
                mentions_competing = True
                break

    payload = {
        "job_title_original": row.get("job_title_original"),
        "major_role_group": row.get("major_role_group"),
        "grouping_justification": just[:1200],
        "crosswalk_overall_risk": row.get("crosswalk_overall_risk"),
        "crosswalk_top_match_role": row.get("crosswalk_top_match_role"),
        "alt_roles": row.get("alt_roles", []),
        "competing_terms": competing_terms or [],
    }

    user_prompt = json.dumps(payload, ensure_ascii=False)

    txt = generate_text(JUDGE_SYSTEM, user_prompt, max_new_tokens=300, temperature=0.0)

    # Best-effort JSON extraction
    m = re.search(r"\{.*\}", txt, re.S)
    if not m:
        return {"hedging_language": hedging, "title_only_reasoning": title_only,
                "mentions_competing_role_terms": mentions_competing, "notes": "Model output not JSON; used heuristics."}

    try:
        out = json.loads(m.group(0))
    except Exception:
        return {"hedging_language": hedging, "title_only_reasoning": title_only,
                "mentions_competing_role_terms": mentions_competing, "notes": "JSON parse failed; used heuristics."}

    # Ensure fields exist
    out.setdefault("hedging_language", hedging)
    out.setdefault("title_only_reasoning", title_only)
    out.setdefault("mentions_competing_role_terms", mentions_competing)
    out.setdefault("notes", "")
    return out

# Example usage:
# 1) Load a local model (if using LOCAL_TRANSFORMERS)
# load_local_model(MODEL_ID, use_4bit=True)
#
# 2) Pick a subset (e.g., top 50 highest deterministic risk)
# subset = scored_df.sort_values("likelihood_error_score_0_5", ascending=False).head(50)
#
# 3) Run judge and merge results
# judged = []
# for _, r in subset.iterrows():
#     judged.append(judge_record(r.to_dict(), competing_terms=[]))
# judged_df = pd.DataFrame(judged)
# judged_df


In [ ]:
# ==== 11) Export scored outputs ====
import os, json

OUTDIR = "/content/output_likelihood_error"
os.makedirs(OUTDIR, exist_ok=True)

csv_path = os.path.join(OUTDIR, "Job_Classifications_Batch_with_Likelihood_Error.csv")
json_path = os.path.join(OUTDIR, "Job_Classifications_Batch_with_Likelihood_Error.json")

df[cols_out].to_csv(csv_path, index=False)

# Attach score back onto original records
records = df_jobs.copy()
records["job_title_key"] = records[TITLE_COL].map(norm)
score_map = df.set_index("job_title_key")["likelihood_error_score_0_5"].to_dict()
band_map  = df.set_index("job_title_key")["likelihood_band"].astype(str).to_dict()
records["likelihood_error_score_0_5"] = records["job_title_key"].map(score_map)
records["likelihood_band"] = records["job_title_key"].map(band_map)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records.drop(columns=["job_title_key"]).to_dict(orient="records"), f, ensure_ascii=False, indent=2)

print("✅ Wrote:")
print(" -", csv_path)
print(" -", json_path)

from google.colab import files
files.download(csv_path)
files.download(json_path)
